# Thermo Scientific Multidrop Combi

The Multidrop Combi is a peristaltic pump reagent dispenser for bulk dispensing into 96-, 384-, and 1536-well plates. It communicates via RS232/USB serial at 9600 baud.

PLR exposes it as a {class}`~pylabrobot.thermo_fisher.multidrop_combi.multidrop_combi.MultidropCombi` device with a {class}`~pylabrobot.capabilities.bulk_dispensers.peristaltic.peristaltic.PeristalticDispensing` capability.

## Setup

In [ ]:
from pylabrobot.thermo_fisher.multidrop_combi import MultidropCombi

md = MultidropCombi(port="/dev/ttyUSB0")  # replace with your port
await md.setup()

On connect, the driver enters remote control mode and retrieves instrument info:

In [ ]:
info = md.driver.get_version()
print(f"{info['instrument_name']} FW {info['firmware_version']} SN {info['serial_number']}")

## Plate configuration

The Multidrop has 10 factory plate types (indexed 0--9). Use `plate_to_type_index` to map a PLR plate to the best-fit factory type, or `plate_to_pla_params` to define a custom plate.

In [ ]:
from pylabrobot.resources.corning.plates import Cor_96_wellplate_360ul_Fb
from pylabrobot.thermo_fisher.multidrop_combi import plate_to_type_index, plate_to_pla_params

plate = Cor_96_wellplate_360ul_Fb("my_plate")

# Try factory type first
try:
    type_idx = plate_to_type_index(plate)
    print(f"Factory plate type: {type_idx}")
except ValueError:
    # No factory match -- define a custom plate
    pla_params = plate_to_pla_params(plate)
    await md.peristaltic.backend.define_plate(**pla_params)
    print(f"Custom plate defined: {pla_params}")

## Cassette type

Set the cassette type before dispensing. The Multidrop supports Standard (0), Small (1), and two user-defined types (2--3).

In [ ]:
from pylabrobot.thermo_fisher.multidrop_combi import CassetteType

await md.peristaltic.backend.set_cassette_type(CassetteType.STANDARD)

## Priming

Prime the hoses before dispensing to fill the tubing with reagent. Use {class}`~pylabrobot.thermo_fisher.multidrop_combi.peristaltic_dispensing_backend.MultidropCombiPeristalticDispensingBackend.PrimeParams` for device-specific settings like prime mode.

In [ ]:
await md.peristaltic.prime(plate=plate, volume=500.0)  # 500 uL

## Dispensing

Dispense to the plate. Pass a `volumes` dict mapping 1-indexed column numbers to volumes in uL.

In [ ]:
# 10 uL to all 12 columns
await md.peristaltic.dispense(
    plate=plate,
    volumes={col: 10.0 for col in range(1, 13)},
)

Use {class}`~pylabrobot.thermo_fisher.multidrop_combi.peristaltic_dispensing_backend.MultidropCombiPeristalticDispensingBackend.DispenseParams` for device-specific settings like plate type, dispensing height, pump speed, and dispensing order:

In [ ]:
from pylabrobot.thermo_fisher.multidrop_combi import (
    MultidropCombiPeristalticDispensingBackend,
    DispensingOrder,
)

await md.peristaltic.dispense(
    plate=plate,
    volumes={col: 25.0 for col in range(1, 13)},
    backend_params=MultidropCombiPeristalticDispensingBackend.DispenseParams(
        plate_type=0,
        dispensing_height=2000,
        pump_speed=50,
        dispensing_order=DispensingOrder.COLUMN_WISE,
    ),
)

Different volumes per column:

In [ ]:
await md.peristaltic.dispense(
    plate=plate,
    volumes={1: 10.0, 2: 20.0, 3: 30.0},
)

## Shaking

The Multidrop has a built-in plate shaker. Shake is a blocking command that returns when done.

In [ ]:
await md.peristaltic.backend.shake(time=3.0, distance=2, speed=10)  # 3s, 2mm, 10Hz

## Purging

Purge (empty) the hoses after dispensing to clear the tubing. Use {class}`~pylabrobot.thermo_fisher.multidrop_combi.peristaltic_dispensing_backend.MultidropCombiPeristalticDispensingBackend.PurgeParams` for device-specific settings like empty mode.

In [ ]:
await md.peristaltic.purge(plate=plate, volume=500.0)

## Moving the plate out

Eject the plate to the loading position.

In [ ]:
await md.peristaltic.backend.move_plate_out()

## Queries

Query instrument parameters and error logs via {class}`~pylabrobot.thermo_fisher.multidrop_combi.driver.MultidropCombiDriver`.

In [ ]:
params = await md.driver.report_parameters()
for line in params[:5]:
    print(line)

In [ ]:
errors = await md.driver.read_error_log()
print(errors)

## Teardown

In [ ]:
await md.stop()